# 🦎 Galápagos Wildlife Classifier — Train & Export TFLite

**Objetivo**: Entrenar un modelo MobileNetV3Small fine-tuned sobre ~40 especies de Galápagos,
exportarlo a TFLite (float16) para uso 100% offline en la app Flutter.

**Fuente de datos**: iNaturalist API (observaciones research-grade, licencia CC-BY/CC0)

**Salida**:
- `galapagos_classifier.tflite` → copiar a `assets/ml/`
- `labels.txt` → copiar a `assets/ml/`

**Runtime requerido**: GPU (T4 gratuito en Colab) — Runtime → Change runtime type → T4 GPU

---
⚠️ **Antes de ejecutar**: Runtime → Change runtime type → GPU (T4)

## 1. Instalar dependencias

In [ ]:
# Colab ya incluye TensorFlow preinstalado (2.17+)
# Solo instalamos lo que puede faltar
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

ensure('Pillow', 'PIL')
ensure('requests')
ensure('tqdm')

import tensorflow as tf
print(f'TensorFlow {tf.__version__}  — OK')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')
assert tf.__version__ >= '2.16', f'Se necesita TF >= 2.16, encontrado {tf.__version__}'

In [ ]:
import os, json, time, shutil, random
import requests
import numpy as np
from pathlib import Path
from tqdm import tqdm
from PIL import Image, UnidentifiedImageError
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'TensorFlow {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 2. Especies objetivo

40 especies visualmente más distintivas de Galápagos.
El notebook busca el taxon_id en iNaturalist automáticamente.

In [ ]:
# (scientific_name, label_en, label_es)
SPECIES = [
    # Reptiles
    ('Amblyrhynchus cristatus',          'Marine Iguana',              'Iguana Marina'),
    ('Conolophus subcristatus',          'Galapagos Land Iguana',      'Iguana Terrestre'),
    ('Conolophus pallidus',              'Santa Fe Land Iguana',       'Iguana de Santa Fe'),
    ('Conolophus marthae',               'Pink Land Iguana',           'Iguana Rosada'),
    ('Chelonoidis porteri',              'Santa Cruz Giant Tortoise',  'Tortuga Gigante'),
    ('Microlophus albemarlensis',        'Galapagos Lava Lizard',      'Lagartija de Lava'),
    ('Pseudalsophis biserialis',         'Galapagos Racer',            'Culebra de Galapagos'),
    # Birds
    ('Sula nebouxii',                    'Blue-footed Booby',          'Piquero Patas Azules'),
    ('Sula granti',                      'Nazca Booby',                'Piquero de Nazca'),
    ('Sula sula',                        'Red-footed Booby',           'Piquero Patas Rojas'),
    ('Fregata magnificens',              'Magnificent Frigatebird',    'Fragata Magnifica'),
    ('Fregata minor',                    'Great Frigatebird',          'Fragata Real'),
    ('Spheniscus mendiculus',            'Galapagos Penguin',          'Pinguino de Galapagos'),
    ('Phoebastria irrorata',             'Waved Albatross',            'Albatros de Galapagos'),
    ('Buteo galapagoensis',              'Galapagos Hawk',             'Gavillan de Galapagos'),
    ('Phoenicopterus ruber',             'American Flamingo',          'Flamingo Americano'),
    ('Nannopterum harrisi',              'Flightless Cormorant',       'Cormoran no Volador'),
    ('Creagrus furcatus',                'Swallow-tailed Gull',        'Gaviota Cola Bifurcada'),
    ('Pelecanus occidentalis',           'Brown Pelican',              'Pelicano Pardo'),
    ('Ardea herodias',                   'Great Blue Heron',           'Garza Azul'),
    ('Butorides sundevalli',             'Lava Heron',                 'Garza de Lava'),
    ('Asio flammeus',                    'Short-eared Owl',            'Buho Orejas Cortas'),
    ('Mimus parvulus',                   'Galapagos Mockingbird',      'Cucuve de Galapagos'),
    ('Zenaida galapagoensis',            'Galapagos Dove',             'Paloma de Galapagos'),
    ('Geospiza fortis',                  'Medium Ground Finch',        'Pinzon Terrestre Mediano'),
    ('Geospiza fuliginosa',              'Small Ground Finch',         'Pinzon Terrestre Pequeno'),
    ('Geospiza magnirostris',            'Large Ground Finch',         'Pinzon Terrestre Grande'),
    ('Pyrocephalus nanus',               'Galapagos Vermilion Flycatcher', 'Brujo de Galapagos'),
    ('Myiarchus magnirostris',           'Galapagos Flycatcher',       'Papamoscas de Galapagos'),
    ('Leucophaeus fuliginosus',          'Lava Gull',                  'Gaviota de Lava'),
    # Mammals
    ('Zalophus wollebaeki',              'Galapagos Sea Lion',         'Lobo Marino'),
    ('Arctocephalus galapagoensis',      'Galapagos Fur Seal',         'Lobo Peletero'),
    # Marine
    ('Chelonia mydas',                   'Green Sea Turtle',           'Tortuga Marina Verde'),
    ('Grapsus grapsus',                  'Sally Lightfoot Crab',       'Cangrejo Zayapa'),
    ('Aetobatus narinari',               'Spotted Eagle Ray',          'Raya Aguila Moteada'),
    ('Mobula birostris',                 'Giant Manta Ray',            'Mantarraya Gigante'),
    ('Rhincodon typus',                  'Whale Shark',                'Tiburon Ballena'),
    ('Sphyrna lewini',                   'Hammerhead Shark',           'Tiburon Martillo'),
    ('Carcharhinus galapagensis',        'Galapagos Shark',            'Tiburon de Galapagos'),
    ('Triaenodon obesus',                'Whitetip Reef Shark',        'Tiburon Punta Blanca'),
]

print(f'Total de clases: {len(SPECIES)}')

## 3. Buscar taxon IDs en iNaturalist

In [ ]:
INAT_API = 'https://api.inaturalist.org/v1'
GALAPAGOS_PLACE_ID = 37123  # Galápagos Islands place_id en iNaturalist

def get_taxon_id(scientific_name: str) -> int | None:
    """Busca el taxon_id de iNaturalist para una especie."""
    resp = requests.get(
        f'{INAT_API}/taxa',
        params={'q': scientific_name, 'rank': 'species,subspecies', 'per_page': 1},
        timeout=15
    )
    if resp.status_code != 200:
        return None
    results = resp.json().get('results', [])
    if results:
        return results[0]['id']
    return None

print('Buscando taxon IDs...')
taxon_ids = {}
for sci_name, en, es in SPECIES:
    tid = get_taxon_id(sci_name)
    taxon_ids[sci_name] = tid
    status = f'✓ {tid}' if tid else '✗ no encontrado'
    print(f'  {sci_name:45s} {status}')
    time.sleep(0.3)  # respetar rate limit

found = sum(1 for v in taxon_ids.values() if v)
print(f'\n{found}/{len(SPECIES)} taxon IDs encontrados')

## 4. Descargar imágenes de iNaturalist

- Máximo `MAX_PER_CLASS` imágenes por especie
- Solo observaciones research-grade con foto
- Descarga la imagen en tamaño `medium` (hasta 500px)

In [ ]:
MAX_PER_CLASS = 150   # aumentar si tienes más tiempo
DATA_DIR = Path('/content/data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {'User-Agent': 'GalapagosWildlifeApp/1.0 (educational; offline species ID)'}

def fetch_observation_photos(taxon_id: int, max_photos: int = 150) -> list[str]:
    """Devuelve URLs de fotos para un taxon_id."""
    urls = []
    page = 1
    while len(urls) < max_photos:
        resp = requests.get(
            f'{INAT_API}/observations',
            params={
                'taxon_id': taxon_id,
                'quality_grade': 'research',
                'photos': 'true',
                'per_page': 50,
                'page': page,
                'order_by': 'votes',  # las más populares primero
            },
            headers=HEADERS,
            timeout=20
        )
        if resp.status_code != 200:
            break
        data = resp.json()
        results = data.get('results', [])
        if not results:
            break
        for obs in results:
            for photo in obs.get('photos', [])[:1]:  # 1 foto por observacion
                url = photo.get('url', '').replace('square', 'medium')
                if url:
                    urls.append(url)
                    if len(urls) >= max_photos:
                        break
            if len(urls) >= max_photos:
                break
        page += 1
        time.sleep(0.5)
    return urls


def download_image(url: str, dest: Path) -> bool:
    """Descarga una imagen y verifica que sea válida."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        if resp.status_code != 200:
            return False
        img = Image.open(__import__('io').BytesIO(resp.content)).convert('RGB')
        if img.width < 100 or img.height < 100:
            return False
        img.save(dest, 'JPEG', quality=90)
        return True
    except Exception:
        return False


# Descargar
download_stats = {}
for sci_name, en_name, _ in SPECIES:
    taxon_id = taxon_ids.get(sci_name)
    if not taxon_id:
        print(f'⚠ Sin taxon_id para {sci_name}, saltando')
        download_stats[sci_name] = 0
        continue

    class_dir = DATA_DIR / sci_name.replace(' ', '_')
    class_dir.mkdir(exist_ok=True)

    # Evitar re-descargar si ya existen suficientes
    existing = list(class_dir.glob('*.jpg'))
    if len(existing) >= MAX_PER_CLASS:
        print(f'✓ {en_name:40s} ya tiene {len(existing)} imágenes')
        download_stats[sci_name] = len(existing)
        continue

    print(f'⬇ {en_name:40s}', end=' ')
    urls = fetch_observation_photos(taxon_id, MAX_PER_CLASS)

    ok = 0
    for i, url in enumerate(tqdm(urls, desc=en_name[:20], leave=False)):
        dest = class_dir / f'{i:04d}.jpg'
        if dest.exists():
            ok += 1
            continue
        if download_image(url, dest):
            ok += 1
        time.sleep(0.1)

    download_stats[sci_name] = ok
    print(f'→ {ok} imágenes')

print('\n=== Resumen de descarga ===')
total = 0
for sci, count in download_stats.items():
    flag = '✓' if count >= 30 else ('⚠' if count >= 10 else '✗')
    print(f'  {flag} {sci:45s} {count:3d} imgs')
    total += count
print(f'Total: {total} imágenes')

## 5. Preparar dataset (train / val split 80/20)

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
TRAIN_DIR = Path('/content/data/train')
VAL_DIR   = Path('/content/data/val')

# Construir lista de clases válidas (min 20 imágenes)
valid_classes = []
for sci_name, en_name, _ in SPECIES:
    class_dir = DATA_DIR / sci_name.replace(' ', '_')
    imgs = list(class_dir.glob('*.jpg'))
    if len(imgs) >= 20:
        valid_classes.append((sci_name, en_name, imgs))
    else:
        print(f'⚠ {sci_name} solo tiene {len(imgs)} imgs — excluida del entrenamiento')

print(f'\nClases válidas para entrenamiento: {len(valid_classes)}')

# Copiar en estructura train/val
for sci_name, en_name, imgs in valid_classes:
    random.shuffle(imgs)
    split = int(len(imgs) * 0.8)
    splits = [('train', imgs[:split]), ('val', imgs[split:])]
    for split_name, split_imgs in splits:
        dest_dir = Path(f'/content/data/{split_name}') / sci_name.replace(' ', '_')
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img_path in split_imgs:
            shutil.copy(img_path, dest_dir / img_path.name)

# Dataset con augmentación
train_ds = keras.utils.image_dataset_from_directory(
    '/content/data/train',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=42,
)
val_ds = keras.utils.image_dataset_from_directory(
    '/content/data/val',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
)

CLASS_NAMES = train_ds.class_names  # nombres de subdirectorios = scientific_name_
NUM_CLASSES = len(CLASS_NAMES)
print(f'Clases: {NUM_CLASSES}, train batches: {len(train_ds)}, val batches: {len(val_ds)}')

# Augmentación + normalización
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name='augmentation')

# MobileNetV3 espera inputs en [-1, 1]
normalize = layers.Rescaling(scale=1.0/127.5, offset=-1.0)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(
    lambda x, y: (normalize(augment(x, training=True)), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)
val_ds = val_ds.map(
    lambda x, y: (normalize(x), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

## 6. Construir modelo — MobileNetV3Small + Transfer Learning

In [ ]:
base_model = keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    include_preprocessing=False,  # normalizamos nosotros
)
base_model.trainable = False  # fase 1: solo entrenar cabeza

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input_1')
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)

model = keras.Model(inputs, outputs, name='galapagos_classifier')
model.summary(expand_nested=False)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

## 7. Entrenar — Fase 1: cabeza (10 épocas) + Fase 2: fine-tune (20 épocas)

In [ ]:
# ─── Fase 1: entrenar cabeza ───────────────────────────────────────
print('=== FASE 1: entrenar cabeza (base congelada) ===')
hist1 = model.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    ]
)

# ─── Fase 2: fine-tune últimas capas ──────────────────────────────
print('\n=== FASE 2: fine-tune (últimas 40 capas del backbone) ===')
base_model.trainable = True
# Descongelar solo las últimas 40 capas
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # LR muy bajo para fine-tune
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

hist2 = model.fit(
    train_ds,
    epochs=20,
    validation_data=val_ds,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
        keras.callbacks.ModelCheckpoint('/content/best_model.keras', save_best_only=True),
    ]
)

## 8. Evaluar

In [ ]:
loss, acc = model.evaluate(val_ds)
print(f'Val accuracy: {acc*100:.1f}%  —  Val loss: {loss:.4f}')

# Matriz de confusión (top errores)
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true, y_pred = [], []
for x_batch, y_batch in val_ds:
    preds = model.predict(x_batch, verbose=0)
    y_true.extend(y_batch.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

# Nombres cortos para el gráfico
short_names = [cn.split('_')[1] if '_' in cn else cn for cn in CLASS_NAMES]

print('\n=== Top clases con más errores ===')
cm = confusion_matrix(y_true, y_pred)
np.fill_diagonal(cm, 0)
error_counts = cm.sum(axis=1)
top_errors = np.argsort(error_counts)[::-1][:10]
for idx in top_errors:
    if error_counts[idx] > 0:
        print(f'  {CLASS_NAMES[idx]:45s} {error_counts[idx]} errores')

## 9. Exportar a TFLite (float16 — buen balance tamaño/precisión)

In [ ]:
# Cargar el mejor modelo guardado
best_model = keras.models.load_model('/content/best_model.keras')

# Convertir a TFLite con cuantización float16
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
# Requerido para operaciones custom de MobileNetV3
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

MODEL_PATH = '/content/galapagos_classifier.tflite'
with open(MODEL_PATH, 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
print(f'Modelo exportado: {size_mb:.1f} MB → {MODEL_PATH}')

# Verificar con intérprete
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
in_det  = interpreter.get_input_details()[0]
out_det = interpreter.get_output_details()[0]
print(f'Input  shape: {in_det["shape"]}  dtype: {in_det["dtype"]}')
print(f'Output shape: {out_det["shape"]}  dtype: {out_det["dtype"]}')

## 10. Generar labels.txt

Formato: una línea por clase = `scientific_name|common_name_en|common_name_es`

El orden debe coincidir con el índice de salida del modelo.

In [ ]:
# Mapear scientific_name (como subdirectorio) → nombres
name_map = {sci.replace(' ', '_'): (sci, en, es) for sci, en, es in SPECIES}

LABELS_PATH = '/content/labels.txt'
with open(LABELS_PATH, 'w') as f:
    for cls_dir in CLASS_NAMES:  # CLASS_NAMES está en el orden del modelo
        if cls_dir in name_map:
            sci, en, es = name_map[cls_dir]
        else:
            sci = cls_dir.replace('_', ' ')
            en = sci
            es = sci
        f.write(f'{sci}|{en}|{es}\n')

with open(LABELS_PATH) as f:
    lines = f.readlines()

print(f'labels.txt generado con {len(lines)} clases:')
for i, line in enumerate(lines):
    print(f'  [{i:02d}] {line.strip()}')

## 11. Descargar archivos para Flutter

Descarga los dos archivos a tu máquina, luego cópialos a `assets/ml/` en el proyecto Flutter.

In [ ]:
from google.colab import files

print('Descargando galapagos_classifier.tflite...')
files.download(MODEL_PATH)

print('Descargando labels.txt...')
files.download(LABELS_PATH)

print()
print('=== INSTRUCCIONES FINALES ===')
print('1. Copia galapagos_classifier.tflite → assets/ml/')
print('2. Copia labels.txt                  → assets/ml/')
print('3. La app usará estos archivos offline automáticamente')
print(f'   Tamaño del modelo: {size_mb:.1f} MB')
print(f'   Clases: {NUM_CLASSES}')
print(f'   Val accuracy: {acc*100:.1f}%')

## 12. (Opcional) Validación manual

In [ ]:
import io

def predict_tflite(image_path: str, top_k: int = 5):
    """Prueba el modelo TFLite con una imagen local."""
    interp = tf.lite.Interpreter(model_path=MODEL_PATH)
    interp.allocate_tensors()
    in_d  = interp.get_input_details()[0]
    out_d = interp.get_output_details()[0]

    img = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32)
    arr = (arr / 127.5) - 1.0  # normalizar igual que en entrenamiento
    arr = np.expand_dims(arr, 0)
    if in_d['dtype'] == np.float16:
        arr = arr.astype(np.float16)

    interp.set_tensor(in_d['index'], arr)
    interp.invoke()
    probs = interp.get_tensor(out_d['index'])[0]

    top_indices = np.argsort(probs)[::-1][:top_k]
    with open(LABELS_PATH) as f:
        labels = [l.strip().split('|') for l in f.readlines()]

    print(f'Predicciones para: {image_path}')
    for idx in top_indices:
        sci, en, es = labels[idx] if idx < len(labels) else (str(idx), str(idx), str(idx))
        print(f'  {probs[idx]*100:5.1f}%  {en} ({sci})')

# Ejemplo: descomentar y cambiar el path
# predict_tflite('/content/data/raw/Sula_nebouxii/0001.jpg')